# FraudGuard AI — Análise Exploratória de Dados (EDA), Comparação de Alternativas e Mapa de Riscos
**Contexto:** Mini Capstone — Pós-Graduação em Engenharia de Inteligência Artificial

Este notebook documenta:
1. **Auditoria de Qualidade do Dataset (`creditcard_2023.csv`)**: reconciliação de 568.630 registros, 31 colunas, ausência de valores nulos, distribuição de classes (50%/50%) e auditoria de duplicatas.
2. **Cuidados Metodológicos e Prevenção de Vazamento (*Data Leakage*)**: exclusão estrita de `id` e `Class` da matriz de atributos `X` (`FEATURES = ["V1", ..., "V28", "Amount"]`).
3. **Comparação Explícita de Alternativas (Custo, Eficácia e Facilidade de Implementação)**:
   - **Alternativa A:** Motor de Regras Determinísticas / Heurísticas Estáticas (`evaluate_rule_based_heuristic`)
   - **Alternativa B:** ML Supervisionado Linear (`Pipeline(StandardScaler, LogisticRegression)`)
   - **Alternativa C:** ML Supervisionado Ensemble (`Pipeline(StandardScaler, RandomForestClassifier)` — **Escolhido**)
   - **Alternativa D:** Modelos de Linguagem de Grande Porte (LLMs / IA Generativa)
4. **Mapa Abrangente de Riscos de Implementação (Software, Dados/ML e Riscos Específicos de LLMs)**.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from ml.src.evaluation import evaluate_rule_based_heuristic
from ml.src.ingestion import validate_csv_in_chunks
from ml.src.preprocessing import FEATURES, stratified_train_val_test_split
from ml.src.training import run_experiments

In [ ]:
csv_path = PROJECT_ROOT / "creditcard_2023.csv"
summary = validate_csv_in_chunks(csv_path, chunksize=20_000)
print("Resumo de Validação em Chunks:")
print(summary)

In [ ]:
df = pd.read_csv(csv_path)
split = stratified_train_val_test_split(df, random_state=42)
rule_baseline = evaluate_rule_based_heuristic(split.X_test, split.y_test)
print("Métricas da Alternativa A (Motor de Regras Heurísticas no Teste Intocado):")
print(rule_baseline)

In [ ]:
artifacts = run_experiments(
    csv_path=csv_path,
    output_dir=PROJECT_ROOT / "ml" / "artifacts",
    parquet_path=PROJECT_ROOT / "ml" / "data" / "creditcard_2023.parquet",
    random_state=42,
)
for version, art in artifacts.items():
    print(f"=== Modelo {version} ({art.algorithm}) ===")
    print("Threshold selecionado (Validação):", art.threshold)
    print("Métricas de Teste Intocado (15%):", art.metrics["test"])

## Síntese da Escolha Arquitetural e Mapa de Riscos (Incluindo LLMs)

- **Eficácia:** Regras estáticas atingiram `F1 = 0,8580` (perdendo 10.056 fraudes no teste); `LogisticRegression` elevou para `F1 = 0,9644`; `RandomForestClassifier` alcançou `F1 = 0,9943` e `PR-AUC = 0,9998`, reduzindo falsos positivos em **83,3%** e falsos negativos em **84,2%**.
- **Custo e Latência vs. LLMs:** Enquanto os pipelines Scikit-learn executam localmente em CPU a custo marginal zero (`0,65 ms` a `16,99 ms`), o uso de LLMs no caminho crítico custaria **US$ 1.137 a US$ 5.686 por lote de 568k transações**, com latência `> 1.500 ms` e alto risco de **alucinação semântica** sobre os atributos anonimizados `V1..V28` (**R-LLM-01** a **R-LLM-06**).